# Example 3: FFT, PSD, and autocorrelation of a speckle field

**Goal.** Load an experimental speckle frame, apply flat-field correction, and inspect the speckle field in the spatial-frequency and lag domains.

This notebook is the frequency-domain companion to Example 2.  
Rather than computing summary metrics, we directly visualise three complementary quantities:

- the **FFT magnitude**,
- the **power spectral density (PSD)**,
- the **autocorrelation**.


In [ ]:
__author__ = ['Rafael Celestre']
__contact__ = 'rafael.celestre@esrf.eu'
__license__ = 'CECILL-2.1'
__copyright__ = 'ESRF - The European Synchrotron'
__created__ = '2026.03.24'
__changed__ = '2026.03.24'

import logging
import sys

import barc4dip as dip
import numpy as np

logging.basicConfig(level=logging.INFO, format="%(message)s")

print(sys.executable)
print(sys.version)

%load_ext autoreload
%autoreload 2

# %matplotlib widget
# %matplotlib inline

## 1) Load a speckle frame

Replace `data_path` with the location of the example dataset.  
Later this can point to the reproducible dataset archived in Zenodo.


In [ ]:
data_path = "/Users/celestre/work/experimental_data/speckle_tracking/speckle_tracking/speckles.h5"

frame = dip.read_image(data_path, image_number=0, verbose=True)
frame.shape

In [ ]:
data_path = "/Users/celestre/work/experimental_data/speckle_tracking/speckle_tracking/darks.h5"

darks = dip.read_image(data_path, mean=True, verbose=True)
darks.shape

In [ ]:
data_path = "/Users/celestre/work/experimental_data/speckle_tracking/speckle_tracking/flats.h5"

flats = dip.read_image(data_path, mean=True, verbose=True)
flats.shape

## 2) Apply flat-field correction

For experimental speckle data, correcting detector offsets and illumination non-uniformity is usually worth doing before any spectral inspection.


In [ ]:
frame = dip.preprocessing.flat_field_correction(frame, flats=flats, darks=darks, verbose=True)

## 3) Quick visual check

A fast image + histogram view is useful before moving to the frequency-domain plots.


In [ ]:
def show_image_summary(
    image: np.ndarray,
    *,
    title: str | None = None,
    bin_min: int = 0,
    bin_max: int = 65535,
    hist: bool = True,
) -> None:
    pmin, pmax = 1.0, 99.0

    vmin = np.nanpercentile(image, pmin)
    vmax = np.nanpercentile(image, pmax)

    dip.plotting.plt_image(
        image,
        title=title,
        vmin=vmin,
        vmax=vmax,
        cmap="srw",
        display_origin="lower",
    )

    if hist:
        dip.plotting.plt_histogram(
            image,
            logy=True,
            cumulative=True,
            percentiles=(1, 99),
            bin_min=bin_min,
            bin_max=bin_max,
        )

    dip.plotting.show()


show_image_summary(frame, bin_min=100, bin_max=25000, title="flat-field corrected frame")

## 4) Pad to a square frame

We pad the corrected image to a **square shape** before computing the FFT, PSD, and autocorrelation.

This is useful here because it makes the sampling along both image directions consistent, so that \(f_x\) and \(f_y\) share the same spacing.  
That, in turn, gives a cleaner and more meaningful **radial average**.

We use the same padded frame for the autocorrelation as well, so the lag axes are treated consistently.


In [ ]:
pframe = dip.geometry.masks.pad_to_square(frame, fill_value=np.mean(frame))
show_image_summary(pframe, bin_min=100, bin_max=25000, title="padded square frame", hist=False)

## 5) FFT magnitude

The Fourier transform is complex.  
Here we display its **magnitude**, which shows how strongly different spatial frequencies contribute to the speckle field.


In [ ]:
fft, fx, fy = dip.signal.fft2d(pframe)
df = fx[1] - fx[0]
fft.shape

In [ ]:
dip.plotting.plt_spectrum2d(
    fft,
    x=fx,
    y=fy,
    log_intensity=False,
    mask_center=True,
    show_phase=False,
    xlabel="$f_x$ (1/px)",
    ylabel="$f_y$ (1/px)",
    title="FFT magnitude",
)
dip.plotting.show()

In [ ]:
radial_mean_fft, R = dip.maths.radial.radial_mean_interpolated(np.abs(fft))

dip.plotting.plt_spectrum1d(
    radial_mean_fft,
    R * df,
    logx=False,
    logy=True,
    title="radially averaged FFT magnitude",
    xlabel="$f$ (1/px)",
    ylabel="|FFT|",
    mask_center=True,
    cumulative=True,
    percentiles=(1, 99),
)
dip.plotting.show()

## 6) Power spectral density (PSD)

The PSD is proportional to \(|FFT|^2\).  
It is often easier to interpret than the raw FFT magnitude when we want to inspect the spatial-frequency distribution of the speckle field.


In [ ]:
psd, fx, fy = dip.signal.psd2d(pframe, scale=True)
df = fx[1] - fx[0]
psd.shape

In [ ]:
dip.plotting.plt_spectrum2d(
    psd,
    x=fx,
    y=fy,
    log_intensity=False,
    mask_center=True,
    show_phase=False,
    xlabel="$f_x$ (1/px)",
    ylabel="$f_y$ (1/px)",
    title="PSD",
)
dip.plotting.show()

In [ ]:
radial_mean_psd, R = dip.maths.radial.radial_mean_interpolated(psd)

dip.plotting.plt_spectrum1d(
    radial_mean_psd,
    R * df,
    logx=False,
    logy=True,
    title="radially averaged PSD",
    xlabel="$f$ (1/px)",
    ylabel="PSD",
    mask_center=True,
    cumulative=True,
    percentiles=(1, 99),
)
dip.plotting.show()

## 7) Autocorrelation

The autocorrelation gives a direct view of the speckle field in **lag space**.  
It is often the most intuitive representation when thinking about characteristic speckle size.


In [ ]:
corr, xlag, ylag = dip.signal.autocorr2d(pframe)
dlag = xlag[1] - xlag[0]
corr.shape

In [ ]:
dip.plotting.plt_spectrum2d(
    corr,
    x=xlag,
    y=ylag,
    log_intensity=False,
    mask_center=False,
    show_phase=False,
    xlabel="$\\Delta_x$ (px)",
    ylabel="$\\Delta_y$ (px)",
    title="autocorrelation",
    xmin=-125,
    xmax=125,
    ymin=-125,
    ymax=125,
)
dip.plotting.show()

In [ ]:
radial_mean_corr, R = dip.maths.radial.radial_mean_interpolated(corr)

dip.plotting.plt_spectrum1d(
    radial_mean_corr,
    R * dlag,
    logx=False,
    logy=False,
    title="radially averaged autocorrelation",
    xlabel="$\\Delta$ (px)",
    ylabel="autocorrelation",
    mask_center=False,
    cumulative=True,
    xmax=100
)
dip.plotting.show()

## Takeaway

- The **FFT magnitude** shows the spatial-frequency content of the speckle field.
- The **PSD** emphasises how that content is distributed in frequency space.
- The **autocorrelation** gives the complementary lag-space view and is often easier to relate to a characteristic speckle size.

Together, these plots provide a compact visual companion to the summary speckle metrics computed in Example 2.
